# Bookmark -> LLM #1 (pipeline réel, PDF avec bookmarks natifs)

```text
PDF réel (avec bookmarks)
  -> extract_raw_toc 
  -> classify_sections (src/llm/classifier.py) -- une fois pour tout le document
```



##  Configuration


In [ ]:
from pathlib import Path
import sys
import json as json_lib

import pandas as pd

import pymupdf as fitz

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PDF_PATH = r"C:\Users\camelia\Downloads\projets\projet1OCP\data\AUSTCOLD.pdf"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF_PATH:", PDF_PATH)


PROJECT_ROOT: c:\Users\camelia\Downloads\halfpath
PDF_PATH: C:\Users\camelia\Downloads\projets\projet1OCP\data\AUSTCOLD.pdf


In [ ]:
from src.document_structure.extract_raw_pdf_toc import extract_raw_toc
import importlib 
importlib.reload(sys.modules['src.document_structure.bookmark_processor'])
from src.document_structure.bookmark_processor import BookmarkProcessor

raw_toc = extract_raw_toc(PDF_PATH)

processor = BookmarkProcessor()
payloads = processor.process(raw_toc)

payloads_flat = [
    item
    for sublist in payloads
    for item in sublist
]

payloads_df = pd.DataFrame(payloads_flat)

print(len(payloads_df), "entrees de payloads")
payloads_df.head(15)

108 entrees de payloads


,section_id,title,section_number,page_start,level
0,toc1_e0,0.0 A5114-OMI-01 Rev 02 Operating and maintena...,0.0,-1,1
1,toc1_e1,Section 1 Service contact details,1,6,1
2,toc1_e2,0.00 Operating and maintenance instructions,0.00,-1,2
3,toc1_e3,0.01 OMI index A5114,0.01,6,2
4,toc1_e4,1.01 Contact details,1.01,9,2
5,toc1_e5,1.02 Standard warning,1.02,15,2
6,toc1_e6,1.03 Standard safety precautions,1.03,20,2
7,toc1_e7,1.04 Standard warranty & support information,1.04,26,2
8,toc1_e8,1.05 Plan history log,1.05,28,2
9,toc1_e9,1.06 Oil consumption log,1.06,35,2


## Passer le contenu du bookmark au LLM #1

Code identique à `05_toc_ocr_llm1_pipeline.ipynb` (section 5) : un appel à
`classify_sections()` par "région" -- ici une seule, tout le document --
avec le même prompt et le même schéma `SectionClassificationResult` que
la production (`src/llm/classifier.py`, non modifié).


In [39]:
import importlib 
importlib.reload(sys.modules['src.llm.config'])
importlib.reload(sys.modules['src.llm.classifier'])
from src.llm.config import ENABLE_LLM, LLM_MODEL
from src.llm.classifier import classify_sections



results_by_region = []

for payload in payloads:

    print(f"\n=== Bookmark -- {len(payload)} sections candidates ===")

    if not ENABLE_LLM:
        print("OPENAI_API_KEY absente -> appel non effectue. Payload pret :")
        print(json_lib.dumps(payload[:5], ensure_ascii=False, indent=2), "..." if len(payload) > 5 else "")
        results_by_region.append(None)
        continue

    try:
        result = classify_sections(payload)
    except Exception as exc:
        print(f"LLM #1 a echoue pour la region {payload[0]['section_id'].split('_')[0]} : {exc}")
        results_by_region.append(None)
        continue

    print("Sections retenues :", result.selected_sections)
    display(pd.DataFrame([c.model_dump() for c in result.sections]))
    results_by_region.append(result)


CONFIG FILE : C:\Users\camelia\Downloads\halfpath\src\llm\config.py
ENV FILE    : C:\Users\camelia\Downloads\halfpath\src\.env
ENV EXISTS  : True

=== Bookmark -- 108 sections candidates ===
Sections retenues : ['toc1_e14', 'toc1_e15', 'toc1_e16', 'toc1_e17', 'toc1_e18', 'toc1_e19', 'toc1_e20', 'toc1_e21', 'toc1_e22', 'toc1_e23', 'toc1_e24', 'toc1_e25', 'toc1_e26', 'toc1_e27', 'toc1_e28', 'toc1_e29', 'toc1_e30', 'toc1_e31', 'toc1_e35', 'toc1_e36', 'toc1_e43', 'toc1_e46', 'toc1_e47', 'toc1_e51', 'toc1_e52', 'toc1_e53', 'toc1_e54', 'toc1_e55', 'toc1_e56', 'toc1_e57', 'toc1_e105']


,section_id,relevance_score,reason,potential_information
0,toc1_e0,20,The title suggests it contains operating and m...,[]
1,toc1_e1,0,Service contact details are unlikely to contai...,[]
2,toc1_e2,20,"Similar to section 0, it suggests general oper...",[]
3,toc1_e3,10,An index is unlikely to contain any relevant e...,[]
4,toc1_e4,0,Contact details do not provide any relevant eq...,[]
...,...,...,...,...
103,toc1_e103,0,A control system architecture is unlikely to c...,[]
104,toc1_e104,0,An air cooled condenser header drawing is unli...,[]
105,toc1_e105,90,An air cooled condenser name plate typically c...,"[Manufacturer / Brand, Model / Type / Referenc..."
106,toc1_e106,0,An air cooled transport arrangement drawing is...,[]


## Vue d'ensemble : sections retenues




In [ ]:
retenue = []

for result in results_by_region:
    for section in result.sections:
        if section.relevance_score > 80:
            retenue.append(section)
retenue_df = pd.DataFrame([c.model_dump() for c in retenue])

retenue=retenue[:len(retenue)-1]
retenue

,section_id,title,section_number,page_start,level
16,toc1_e16,3.02 Compressor Data Sheet,3.02,107,2
17,toc1_e17,3.03 Compressor motor data sheet,3.03,116,2
18,toc1_e18,3.04 Oil Pump Data Sheet,3.04,155,2
19,toc1_e19,3.05 Oil Pump Motor Data Sheet,3.05,157,2
20,toc1_e20,3.06 Oil Cooler Data Sheet,3.06,160,2
21,toc1_e21,3.07 Oil separator data sheet,3.07,166,2
22,toc1_e22,3.08 Receiver data sheet,3.08,170,2
23,toc1_e23,3.09 Purger data sheet,3.09,171,2
24,toc1_e24,3.10 Economiser Data Sheet,3.10,174,2
25,toc1_e25,3.11 Air Cooled Condenser data sheet,3.11,179,2


In [ ]:

clean_sections = processor.clean_sections   # deja stocke par process(), rien a refaire

#    dans le meme ordre que clean_sections/payloads
# 'retenue' = votre DataFrame filtre (16 lignes, relevance_score > 80)
selected_ids = set(retenue['section_id'])

overview_rows = []
for payload in payloads:              # toutes les sections, toutes regions
    for entry in payload:
        overview_rows.append({
            "section_id": entry["section_id"],
            "title": entry["title"],
            "page": entry["page_start"],
            "retenue": entry["section_id"] in selected_ids,   # True seulement pour vos 16
        })
overview = pd.DataFrame(overview_rows)

print("Total retenues :", int(overview["retenue"].sum()), "/", len(overview))  # doit afficher 16




Total retenues : 15 / 108


In [62]:
from src.ocr.pages_from_sections import PageResolutionPipeline
pipeline = PageResolutionPipeline(
    pdf_path=PDF_PATH,
    raw_toc=raw_toc,
    clean_sections=clean_sections,
    overview=overview,
    processor=processor,
    batch_size=5,
    max_workers=5,
)
pipeline.resolved_df

Recherche de la page pour '3.15 Final stage oil separator data sheet' par lots de 5 pages, entre 193 et 2870...
    lot de pages 193-197 (traitement parallele, 5 thread(s))...
[OCR] moteur utilise pour le haut de page : Tesseract (pytesseract) -- C:\Program Files\Tesseract-OCR\tesseract.exe (TESSDATA_PREFIX=C:\Program Files\Tesseract-OCR\tessdata)
    page 193 [texte natif] vide -> tentative OCR
    page 193 [OCR haut de page / pytesseract] "a EP a a a a a a a J | _SECTION 3.14 _" -> pas de match
    page 194 [texte natif] vide -> tentative OCR
    page 194 [OCR haut de page / pytesseract_psm6] "E 14" -> pas de match
    page 195 [texte natif] "Jord International Pty Ltd 38 Oxley Street St Leonards NSW 2065 Australia T: +61 2 8425 1500 F: +61 2 8425 1555 www.jord.com.au VIBRATION SWITCH DATA SHEET 1 METRIX RV RV 14.03.12 For Approval 0 METRI" -> pas de match
    page 196 [texte natif] vide -> tentative OCR
    page 196 [OCR haut de page / pytesseract] "SPECIFICATIONS Function: Armature

,position,title,page_start,source,page_end
0,16,3.02 Compressor Data Sheet,107,bookmark,115
1,17,3.03 Compressor motor data sheet,116,bookmark,154
2,18,3.04 Oil Pump Data Sheet,155,bookmark,156
3,19,3.05 Oil Pump Motor Data Sheet,157,bookmark,159
4,20,3.06 Oil Cooler Data Sheet,160,bookmark,165
5,21,3.07 Oil separator data sheet,166,bookmark,169
6,22,3.08 Receiver data sheet,170,bookmark,170
7,23,3.09 Purger data sheet,171,bookmark,173
8,24,3.10 Economiser Data Sheet,174,bookmark,178
9,25,3.11 Air Cooled Condenser data sheet,179,bookmark,182


In [63]:
resolved_df=pipeline.resolved_df

In [67]:
resolved_df.to_csv(
    PROJECT_ROOT / "data" / "resolved_sections.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)